In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("healthcare_dataset.csv") 

print("Shape:", df.shape)  
df.head()                  

Shape: (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

In [4]:
df.describe()

,Age,Billing Amount,Room Number
count,55500.000000,55500.000000,55500.000000
mean,51.539459,25539.316097,301.134829
std,19.602454,14211.454431,115.243069
min,13.000000,-2008.492140,101.000000
25%,35.000000,13241.224652,202.000000
50%,52.000000,25538.069376,302.000000
75%,68.000000,37820.508436,401.000000
max,89.000000,52764.276736,500.000000


In [5]:
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percent": missing_percent})

print(missing_df[missing_df["Missing Count"] > 0])

Empty DataFrame
Columns: [Missing Count, Missing Percent]
Index: []


In [8]:
df = df.drop_duplicates()
print("removing duplicates:", df.shape)

removing duplicates: (54966, 15)


In [9]:
# Remove spaces, lowercase all column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Cleaned Column Names:")
print(df.columns.tolist())

Cleaned Column Names:
['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']


In [11]:
# Convert to datetime format
df["date_of_admission"] = pd.to_datetime(df["date_of_admission"])
df["discharge_date"] = pd.to_datetime(df["discharge_date"])

print("Date columns converted")
print(df[["date_of_admission", "discharge_date"]].dtypes)

Date columns converted
date_of_admission    datetime64[ns]
discharge_date       datetime64[ns]
dtype: object


In [12]:
# New column: how many days patient stayed
df["length_of_stay"] = (df["discharge_date"] - df["date_of_admission"]).dt.days

print("Length of Stay column created ✅")
print(df["length_of_stay"].describe())

Length of Stay column created ✅
count    54966.000000
mean        15.499290
std          8.661471
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: length_of_stay, dtype: float64


In [13]:
# Categorize patients by age group
def categorize_age(age):
    if age <= 18:
        return "Paediatric"
    elif age <= 65:
        return "Adult"
    else:
        return "Geriatric"

df["age_category"] = df["age"].apply(categorize_age)

print("Age Category Distribution:")
print(df["age_category"].value_counts())

Age Category Distribution:
age_category
Adult         37984
Geriatric     16096
Paediatric      886
Name: count, dtype: int64


In [15]:
# Useful for trend analysis later
df["admission_year"] = df["date_of_admission"].dt.year
df["admission_month"] = df["date_of_admission"].dt.month_name()

print("Year and Month columns created")
df[["date_of_admission", "admission_year", "admission_month"]].head()

Year and Month columns created


,date_of_admission,admission_year,admission_month
0,2024-01-31,2024,January
1,2019-08-20,2019,August
2,2022-09-22,2022,September
3,2020-11-18,2020,November
4,2022-09-19,2022,September


In [16]:
# Remove extra spaces and fix capitalization
text_columns = ["gender", "blood_type", "medical_condition",
                "admission_type", "medication", "test_results"]

for col in text_columns:
    df[col] = df[col].str.strip().str.title()

print("Text columns standardized")

Text columns standardized


In [23]:
#year and month extracted
df["admission_year"] = df["date_of_admission"].dt.year
df["admission_month"] = df["date_of_admission"].dt.month_name()

print("Year & Month extracted")

Year & Month extracted


In [21]:
# Check for negative or zero billing values
print("Min Billing:", df["billing_amount"].min())
print("Max Billing:", df["billing_amount"].max())

# Round billing amount to 2 decimal places
df["billing_amount"] = df["billing_amount"].round(2)

print("\nBilling Amount cleaned ")

Min Billing: 9.24
Max Billing: 52764.28

Billing Amount cleaned 


In [20]:
# Check negative values
print("Negative billing rows:", (df["billing_amount"] < 0).sum())

# Replace negative values with the median billing amount
median_billing = df[df["billing_amount"] > 0]["billing_amount"].median()
df["billing_amount"] = df["billing_amount"].apply(
    lambda x: median_billing if x < 0 else x
)

# Round to 2 decimal places
df["billing_amount"] = df["billing_amount"].round(2)

print("Billing Amount fixed ")
print("Min Billing:", df["billing_amount"].min())

Negative billing rows: 0
Billing Amount fixed 
Min Billing: 9.24


In [24]:
print("=== FINAL DATASET ===")
print("Shape:", df.shape)
print("\nNew Columns Added:")
print("→ length_of_stay")
print("→ age_category")
print("→ admission_year")
print("→ admission_month")
print("\nAny Missing Values:", df.isnull().sum().sum())

# Save cleaned file
df.to_csv("healthcare_cleaned.csv", index=False)
print("\n Cleaned dataset saved as 'healthcare_cleaned.csv'")

=== FINAL DATASET ===
Shape: (54966, 19)

New Columns Added:
→ length_of_stay
→ age_category
→ admission_year
→ admission_month

Any Missing Values: 0

 Cleaned dataset saved as 'healthcare_cleaned.csv'
